In [ ]:
%pip install -q -U "crewai[google-genai,tools]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.9/238.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 724.

In [ ]:
import os
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "Please add GEMINI_API_KEY to Google Colab Secrets."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(
    model="gemini/gemini-3.1-flash-lite-preview",
    api_key=GEMINI_API_KEY,
    temperature=0.2
)

print("Gemini configured successfully!")

Gemini configured successfully!


In [ ]:
from crewai_tools import FileReadTool, FileWriterTool

file_reader = FileReadTool(file_path="transcript.txt")
file_writer = FileWriterTool()

print("File tools initialized!")

File tools initialized!


In [ ]:
from crewai import Agent

summarizer = Agent(
    role="Meeting Summarizer",
    goal="Summarize the meeting transcript in exactly five concise bullet points.",
    backstory="You are skilled at extracting the most important discussion points from meetings.",
    llm=llm,
    verbose=True
)

action_finder = Agent(
    role="Action Item Extractor",
    goal="Identify all action items, responsible people, and deadlines from the transcript.",
    backstory="You carefully track commitments and never invent missing owners or deadlines.",
    llm=llm,
    verbose=True
)

decision_logger = Agent(
    role="Decision Logger",
    goal="Identify the decisions explicitly made during the meeting.",
    backstory="You maintain accurate decision records and distinguish confirmed decisions from suggestions.",
    llm=llm,
    verbose=True
)

print("Three specialist agents created successfully!")

Three specialist agents created successfully!


In [ ]:
from crewai import Task

summary_task = Task(
    description="""
    Read the following meeting transcript:

    {transcript}

    Summarize the main discussion in exactly five concise bullet points.
    Include important dates, project goals, and key updates.
    Do not add information that is not in the transcript.
    """,
    expected_output="Exactly five concise bullet points summarizing the meeting.",
    agent=summarizer,
    async_execution=True
)

action_task = Task(
    description="""
    Analyze the meeting transcript:

    {transcript}

    Extract every action item. For each item, identify:
    - Task
    - Responsible person
    - Deadline

    If an owner or deadline is not stated, write 'Not specified'.
    Do not invent information.
    """,
    expected_output="A structured list of action items, owners, and deadlines.",
    agent=action_finder,
    async_execution=True
)

decision_task = Task(
    description="""
    Analyze the meeting transcript:

    {transcript}

    List the decisions explicitly agreed upon.
    Distinguish confirmed decisions from suggestions.
    If there are no confirmed decisions, state that clearly.
    """,
    expected_output="A concise list of confirmed meeting decisions.",
    agent=decision_logger,
    async_execution=True
)

print("Three parallel tasks created successfully!")

Three parallel tasks created successfully!


In [ ]:
formatter = Agent(
    role="Meeting Notes Formatter",
    goal="Combine the meeting summary, action items, and decisions into a clear Markdown report.",
    backstory="You create accurate, professional meeting minutes that are easy to read and act upon.",
    llm=llm,
    verbose=True
)

print("Formatter agent created successfully!")

Formatter agent created successfully!


In [ ]:
format_task = Task(
    description="""
    Create complete meeting notes using the outputs from the
    previous tasks provided in your task context.

    Combine the meeting summary, action items, and decisions into
    one clear, professional Markdown report.

    Organize the report using these headings:

    # Meeting Notes
    ## Executive Summary
    ## Key Discussion Points
    ## Action Items
    ## Decisions Made
    ## Upcoming Milestones

    Requirements:
    - Use a Markdown table for action items.
    - Include task, owner, and deadline in the table.
    - Preserve all facts and dates from the transcript.
    - Do not invent missing information.
    - Avoid repeating the same information unnecessarily.
    """,
    expected_output="A complete, well-formatted Markdown meeting notes report.",
    agent=formatter,
    context=[summary_task, action_task, decision_task],
    output_file="notes.md"
)

In [ ]:
from crewai import Crew, Process

meeting_crew = Crew(
    agents=[
        summarizer,
        action_finder,
        decision_logger,
        formatter
    ],
    tasks=[
        summary_task,
        action_task,
        decision_task,
        format_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Meeting Notes Crew created successfully!")

Meeting Notes Crew created successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    print("Please upload a .txt file.")
else:
    filename = next(iter(uploaded))

    if not filename.lower().endswith(".txt"):
        print("Please upload a .txt file.")
    else:
        transcript = uploaded[filename].decode("utf-8").strip()

        if not transcript:
            print("The transcript file is empty.")
        else:
            result = await meeting_crew.kickoff_async(
                inputs={"transcript": transcript}
            )

            print("\nMeeting Notes Crew completed!")
            print(result)

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown

report_path = Path("notes.md")

if report_path.exists() and report_path.stat().st_size > 0:
    report = report_path.read_text(encoding="utf-8")

    print("Report created successfully!")
    print("Location:", report_path.resolve())
    print("Size:", report_path.stat().st_size, "bytes")

    display(Markdown(report))
else:
    print("notes.md is missing or empty.")
    print("Check the execution logs in Cell 10.")

In [ ]:
from google.colab import files

files.download("notes.md")